# MILCCI Demo
**Multi-axis Interpretable Latent Component and Condition Inference**

This notebook demonstrates how to use MILCCI to decompose a 3D tensor
Y (neurons x time x trials) into condition-varying spatial maps A
and temporal traces Phi, with similarity regularization along multiple
label axes.

## Installation
```bash
pip install -e .
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import milcci
from milcci import plotting

print('MILCCI version:', milcci.__version__)

## 1. Generate synthetic data

MILCCI operates on data organized as a 3D tensor of shape
**(N neurons, T time bins, M trials)**.

Each trial carries a multi-axis label, e.g. `(stimulus_id, block_number)`.
MILCCI assigns a subset of ensembles to each axis and enforces that
the spatial map A is shared across trials that agree on that axis.

Here we generate synthetic data with:
- 2 label axes: axis_0 has 3 values (0, 1, 2), axis_1 has 2 values (0, 1)
- 2 ensembles per axis (4 total)
- Temporal traces sampled from Gaussian Processes, so trials sharing
  the same axis value have correlated Phi for those ensembles

In [ ]:
synth = milcci.generate_synthetic_data(
    N=30,                        # neurons
    T=80,                        # time bins
    n_ensembles_each=[2, 2],     # 2 ensembles per axis
    axis_values=[[0, 1, 2], [0, 1]],
    noise_std=0.2,
    trials_per_condition=3,      # 3 trials per unique condition
    gp_length_scale=0.15,        # GP smoothness
    gp_sigma=0.25,               # within-axis-value variability
    seed=42,
)

Y = synth['Y']
labels = synth['labels']
numbers2tuples = synth['numbers2tuples']
N, T, M = Y.shape
n_ensembles = sum(synth['n_ensembles_each'])

print('Data shape: N=%d neurons, T=%d time bins, M=%d trials' % (N, T, M))
print('Unique conditions: %d' % len(numbers2tuples))
print('Ensembles: %d total' % n_ensembles)
print('Condition tuples:', synth['labels_tuples'][:6], '...')

## 2. Run MILCCI

The main function is `milcci.fit()`. Key parameters:

| Parameter | Description |
|-----------|-------------|
| `n_ensembles` | Total number of components |
| `n_ensembles_each` | Components per axis (must sum to n_ensembles) |
| `nu` | Per-ensemble similarity strength |
| `lambda_similarity` | Global regularization weight |
| `split_A` | Infer separate A per axis-value (recommended) |
| `cont_axis_list` | Which axes are continuous (e.g. trial number) |

In [ ]:
result = milcci.fit(
    data=Y,
    labels=labels,
    numbers2tuples=numbers2tuples,
    n_ensembles=n_ensembles,
    n_ensembles_each=synth['n_ensembles_each'],
    nu=[0.01] * n_ensembles,
    lambda_similarity=100,
    factor_A=5,
    decor_A=2,
    num_repeats=15,
    cont_axis_list=[],          # both axes discrete
    params_init_A={'ensemble_positive': False},
    split_A=True,
    verbose=True,
    seed=42,
)

Phi = result['Phi']        # (T, P, M) temporal traces
A = result['A']            # (N, P, K) spatial maps per unique condition
A_full = result['A_full']  # (N, P, M) spatial maps per trial

print('Phi shape:', Phi.shape)
print('A shape:', A.shape)
print('A_full shape:', A_full.shape)

## 3. Evaluate reconstruction quality

In [ ]:
r2 = milcci.global_r2(Y, A_full, Phi)
rho = milcci.reconstruction_correlation(Y, A_full, Phi)
r2_vec = milcci.per_trial_r2(Y, A_full, Phi)

print('Global R^2:          %.4f' % r2)
print('Reconstruction rho:  %.4f' % rho)
print('Per-trial R^2: mean=%.3f, std=%.3f' % (r2_vec.mean(), r2_vec.std()))

## 4. Visualize results

### 4.1 Spatial maps A per condition
Dashed lines separate ensembles assigned to different axes.

In [ ]:
plotting.plot_A_heatmaps(
    A, result['params']['labels_unique_order'], numbers2tuples,
    class_names=synth['class_names'],
    n_ensembles_each=synth['n_ensembles_each'],
)

### 4.2 A similarity across conditions

Conditions sharing the same value on a given axis should have
highly correlated A for the ensembles assigned to that axis.

In [ ]:
plotting.plot_A_similarity_matrix(
    A, result['params']['labels_unique_order'], numbers2tuples,
    n_ensembles_each=synth['n_ensembles_each'],
    class_names=synth['class_names'],
)

### 4.3 Temporal traces Phi

In [ ]:
plotting.plot_phi_traces(
    Phi, labels, result['params']['labels_unique_order'], numbers2tuples,
    n_ensembles_each=synth['n_ensembles_each'],
    class_names=synth['class_names'],
)

### 4.4 Reconstruction vs original

In [ ]:
plotting.plot_reconstruction(Y, A_full, Phi, trial_indices=[0, 6, 12])

### 4.5 Per-trial R^2

In [ ]:
plotting.plot_r2_per_trial(r2_vec, labels=labels, numbers2tuples=numbers2tuples)

### 4.6 Ground-truth comparison (synthetic only)

Compare estimated A and Phi to the known ground truth.

In [ ]:
plotting.plot_ground_truth_comparison(
    A, synth['A_true'], Phi, synth['Phi_true'],
    result['params']['labels_unique_order'],
)

### 4.7 Summary dashboard

In [ ]:
plotting.plot_summary(
    Y, result, numbers2tuples, labels,
    class_names=synth['class_names'],
    n_ensembles_each=synth['n_ensembles_each'],
)

## 5. Validate axis structure

The key property of MILCCI: conditions sharing the same value on
axis k should have identical A for the ensembles assigned to axis k.

In [ ]:
labels_unique = result['params']['labels_unique_order']
for val in synth['axis_values'][0]:
    matching = [i for i, lab in enumerate(labels_unique)
               if numbers2tuples[lab][0] == val]
    if len(matching) > 1:
        A0 = A[:, :2, matching[0]]
        A1 = A[:, :2, matching[1]]
        corr = np.corrcoef(A0.flatten(), A1.flatten())[0, 1]
        print('axis_0 value=%d: A[:,:2] corr between conds %d and %d = %.4f'
              % (val, matching[0], matching[1], corr))